# PulmoGnosis — Etapa 2 pe JupyterHub

Descărcarea datelor, construirea cuburilor, antrenarea modelelor A, B, C și salvarea rezultatelor.
Procedura completă și explicațiile sunt în **PLAN.md, secțiunea 10**.

**Rulați celulele în ordine.** Pașii lungi (descărcarea, antrenarea pe toți pacienții) pornesc **în fundal**,
cu jurnal (*log*): puteți închide browserul, iar celula „Progres" de după ele arată unde au ajuns.

| Pas | Ce face | Durată aprox. |
|---|---|---|
| 0–2 | Mediul, regulile git, căile | 5–10 min |
| 3 | Metadate + cohortă + modelul A | 1 min |
| 4–5 | Selecția și descărcarea NSCLC-Radiogenomics (144 pacienți) | câteva ore (în fundal) |
| 6–7 | Cuburile 128³ + verificarea vizuală | 30–60 min |
| 8–9 | Modelul C (40 pacienți, apoi toți) | ~1–3 h |
| 10 | Modelul B (Kaggle) | ~30 min |
| 11–12 | Tabelul de rezultate + arhiva de descărcat | 1 min |

## 0. Căi și verificări (GPU, spațiu pe disc, ramura git)

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))   # the package is importable even before a kernel restart
DATA = Path.home() / "pulmognosis_data"          # datele stau ÎN AFARA repository-ului
DATA.mkdir(exist_ok=True)
(REPO / "logs").mkdir(exist_ok=True)

# data/ din repository arată spre DATA, ca scripturile să-și găsească fișierele implicite
link = REPO / "data"
if not link.exists():
    link.symlink_to(DATA, target_is_directory=True)

print("Repository:", REPO)
print("Date:      ", DATA)
!git -C {REPO} branch --show-current
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h ~ | tail -1

## 1. Mediul Python

**Kernel-ul trebuie să fie `venv`** (dreapta-sus în bara notebook-ului; altfel *Kernel → Change Kernel… → venv*).
Kernel-ul implicit „Python 3 (ipykernel)" folosește Python-ul sistemului, care refuză instalarea de pachete
(*externally-managed-environment*); `venv` este mediul personal din `~/venv`.

Celula instalează pachetele proiectului. PyTorch se instalează doar dacă lipsește sau nu vede GPU-ul
(versiunea pentru CUDA 12.1; dacă serverul are altă versiune de CUDA, schimbați `cu121`).

In [ ]:
if sys.prefix == sys.base_prefix:
    raise RuntimeError("Kernel-ul folosește Python-ul sistemului. Alegeți kernel-ul 'venv' "
                       "(Kernel → Change Kernel… → venv), apoi rulați din nou celula 0 și această celulă.")
try:
    import torch
    ok = torch.cuda.is_available()
except ImportError:
    ok = False
if not ok:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "torchvision",
                           "--index-url", "https://download.pytorch.org/whl/cu121"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt", "-e", "."])

import torch, monai
print("Python:", sys.executable)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| MONAI", monai.__version__)

## 2. Reguli git locale (ca datele să nu ajungă niciodată în git)

In [ ]:
rules = ["/data/", "/checkpoints/", "/runs/", "/logs/", "/wandb/", "/mlruns/", ".venv/", "__pycache__/",
         "*.py[cod]", ".pytest_cache/", "*.egg-info/", ".env", "*.dcm", "*.nii", "*.nii.gz", "*.npz",
         "*.pt", "*.pth", "*.ckpt", ".ipynb_checkpoints/"]
exclude = REPO / ".git" / "info" / "exclude"
present = exclude.read_text().splitlines() if exclude.exists() else []
with exclude.open("a") as f:
    for r in rules:
        if r not in present:
            f.write(r + "\n")
!git -C {REPO} status --short

## 3. Metadate, cohortă și modelul A (verifică mediul: trebuie să dea aceleași cifre ca în PLAN.md)

In [ ]:
import urllib.request
(DATA / "metadata").mkdir(exist_ok=True)
base = "https://www.cancerimagingarchive.net/wp-content/uploads"
for name in ["NSCLCR01Radiogenomic_DATA_LABELS_2018-05-22_1500-shifted.csv",
             "NSCLC_Radiogenomics-6-1-21-Version-4-nbia-digest.xlsx",
             "statistics-clinical-20201221.xlsx"]:
    target = DATA / "metadata" / name
    if not target.exists():
        urllib.request.urlretrieve(f"{base}/{name}", target)
print(sorted(p.name for p in (DATA / "metadata").iterdir()))

!{sys.executable} scripts/build_cohort.py
!{sys.executable} scripts/train_clinical.py
# Așteptat: 211 pacienți / 144 cu SEG / EGFR 43 din 172; EGFR AUC ≈ 0,72, KRAS ≈ 0,55

## 4. Ce descărcăm (doar afișează dimensiunea — nu descarcă nimic)

Seriile necesare pentru cei 144 de pacienți R01 cu mască tumorală: CT diagnostic, masca SEG,
CT-ul din studiul PET (pentru registrarea din Etapa 3) și PET-ul corectat de atenuare.
Măștile adăugate ulterior de IA în IDC (*AIMI*, care conturează și plămânii) sunt excluse.
**Verificați că sunt 144 de pacienți și că totalul încape în spațiul liber** (buget: PLAN.md §3.6).

In [ ]:
from idc_index import IDCClient
from pulmognosis.data.idc import select_nsclc_radiogenomics, size_report

client = IDCClient()
selection = select_nsclc_radiogenomics(client.index)
print(selection["PatientID"].nunique(), "pacienți")
print(size_report(selection))
selection.to_csv(DATA / "metadata" / "idc_selection_nsclc_radiogenomics.csv", index=False)
!df -h ~ | tail -1

## 5. Descărcarea (în fundal, câteva ore)

Pornește descărcarea și se întoarce imediat. Dacă se oprește (deconectare, server repornit),
rulați din nou celula: fișierele existente sunt sărite.

In [ ]:
script = f'''
import pandas as pd
from idc_index import IDCClient
from pulmognosis.data.idc import download
sel = pd.read_csv("{DATA}/metadata/idc_selection_nsclc_radiogenomics.csv")
download(IDCClient(), sel, "{DATA}/raw/nsclc_radiogenomics")
print("DOWNLOAD FINISHED")
'''
(REPO / "logs" / "download.py").write_text(script)
subprocess.Popen(f"nohup {sys.executable} logs/download.py > logs/download.log 2>&1 &", shell=True, cwd=REPO)
print("Descărcarea a pornit. Urmăriți progresul cu celula următoare.")

In [ ]:
# Progres descărcare: rulați oricând
raw = DATA / "raw" / "nsclc_radiogenomics"
n = len([p for p in raw.iterdir() if p.is_dir()]) if raw.exists() else 0
print(f"{n} / 144 pacienți au început să se descarce")
!du -sh {raw} 2>/dev/null
!tail -n 5 logs/download.log
!df -h ~ | tail -1

## 6. Cuburile 128³ (0,75 mm) în jurul tumorii

Pentru fiecare pacient: masca SEG → CT-ul pe care a fost desenată → centrul tumorii → cubul, salvat în
`DATA/processed/nsclc_radiogenomics/`. Doar CT în Etapa 2 (PET-ul vine după registrare, în Etapa 3).
Rulați după ce descărcarea s-a terminat (`DOWNLOAD FINISHED` în jurnal).

In [ ]:
!{sys.executable} scripts/build_cubes.py --raw {DATA}/raw/nsclc_radiogenomics --out {DATA}/processed/nsclc_radiogenomics 2>&1 | tail -n 15

import pandas as pd
log = pd.read_csv(DATA / "processed" / "nsclc_radiogenomics" / "build_log.csv")
print(log["status"].value_counts())
print(log["ct_choice"].value_counts())
log[log["status"] == "failed"][["patient_id", "error"]]

## 7. Verificare vizuală (obligatorie)

Fiecare cub trebuie să aibă tumora **în centru** (crucea roșie). Notați pacienții la care nu e așa.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from pulmognosis.preprocessing.intensity import window, WINDOWS

cubes = sorted((DATA / "processed" / "nsclc_radiogenomics").glob("*.npz"))
show = cubes[:48]
cols = 8
fig, axes = plt.subplots(int(np.ceil(len(show) / cols)), cols, figsize=(2.2 * cols, 2.3 * np.ceil(len(show) / cols)))
for ax, path in zip(axes.flat, show):
    ct = np.load(path)["ct"].astype(np.float32)
    ax.imshow(window(ct[64], *WINDOWS["lung"]), cmap="gray")
    ax.plot(64, 64, "r+", ms=10)
    ax.set_title(path.stem, fontsize=8)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()
print(f"{len(cubes)} cuburi; schimbați `cubes[:48]` în `cubes[48:96]` etc. pentru restul")

## 8. Modelul C pe date puține (40 de pacienți) — cerința Etapei 2

3D ResNet-18 (greutăți MedicalNet), adenocarcinom vs. scuamos, pe aceleași 5 fold-uri ca modelul A.

In [ ]:
!{sys.executable} scripts/train_model3d.py --cubes {DATA}/processed/nsclc_radiogenomics --subset 40 --out runs/model_c_histology_40 --pretrained medicalnet 2>&1 | grep -v "^  epoch"

## 9. Modelul C pe toți pacienții cu mască (în fundal, ~1–3 h)

In [ ]:
cmd = (f"nohup {sys.executable} scripts/train_model3d.py --cubes {DATA}/processed/nsclc_radiogenomics "
       f"--out runs/model_c_histology_all --pretrained medicalnet > logs/model_c_all.log 2>&1 &")
subprocess.Popen(cmd, shell=True, cwd=REPO)
print("Antrenarea a pornit. Progres: celula următoare.")

In [ ]:
# Progres modelul C (toți pacienții)
!grep -E "^fold|done in|AUC|Final|Saved" logs/model_c_all.log | tail -n 15
!tail -n 2 logs/model_c_all.log
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

## 10. Modelul B — CNN 2D pe setul Kaggle

Necesită cheia API Kaggle: pe kaggle.com → *Settings* → *API* → *Create New Token* descarcă `kaggle.json`;
încărcați-l în folderul *home* din JupyterHub (butonul *Upload*), apoi rulați celula.

In [ ]:
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
if (Path.home() / "kaggle.json").exists():
    shutil.move(str(Path.home() / "kaggle.json"), kaggle_dir / "kaggle.json")
os.chmod(kaggle_dir / "kaggle.json", 0o600)

!kaggle datasets download -d mohamedhanyyy/chest-ctscan-images -p {DATA}/raw/kaggle --unzip -q
!ls {DATA}/raw/kaggle

In [ ]:
kaggle_data = next(p for p in (DATA / "raw" / "kaggle").rglob("train")).parent   # folderul cu train/ valid/ test/
!{sys.executable} scripts/train_cnn2d.py --data {kaggle_data} --out runs/model_b_cnn2d 2>&1 | grep -v "^epoch"

## 11. Tabelul de rezultate (copiați-l în raport — doar aceste cifre, nimic estimat)

In [ ]:
def fmt(m, key="auc"):
    lo, hi = m[f"{key}_ci95"]
    return f"{m[key]:.2f} ({lo:.2f}–{hi:.2f})".replace(".", ",")

rows = []
a = json.loads(Path("runs/model_a_clinical/metrics.json").read_text())
rows.append(("A: regresie logistică clinică", "EGFR", a["egfr"]["patients"], a["egfr"]["positives"], fmt(a["egfr"])))
rows.append(("A: regresie logistică clinică", "KRAS", a["kras"]["patients"], a["kras"]["positives"], fmt(a["kras"])))
for run, label in [("model_c_histology_40", "C: 3D ResNet-18, 40 pacienți"), ("model_c_histology_all", "C: 3D ResNet-18, toți")]:
    p = Path("runs") / run / "metrics.json"
    if p.exists():
        m = json.loads(p.read_text())
        rows.append((label, "scuamos vs. adeno", m["patients"], m["positives"], fmt(m)))
p = Path("runs/model_b_cnn2d/metrics.json")
if p.exists():
    t = json.loads(p.read_text())["test"]
    lo, hi = t["macro_auc_ci95"]
    rows.append(("B: CNN 2D (Kaggle, test)", "4 clase (AUC macro)", t["images"], "—", f"{t['macro_auc_ovr']:.2f} ({lo:.2f}–{hi:.2f})".replace(".", ",")))

table = pd.DataFrame(rows, columns=["Model", "Țintă", "Pacienți/imagini", "Pozitivi", "AUC (IÎ 95%)"])
table

## 12. Salvarea rezultatelor pentru descărcare

Creează `~/pulmognosis_rezultate_etapa2.zip` (metrici, predicții, configurații, jurnale — fără date de pacient
și fără greutăți). Descărcați-l din browserul de fișiere JupyterHub (click dreapta → *Download*) și trimiteți-l
în conversația cu asistentul pentru raport. Greutățile modelelor rămân pe server, în `runs/*/final_model.pt`.

In [ ]:
import zipfile
archive = Path.home() / "pulmognosis_rezultate_etapa2.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for path in list(Path("runs").rglob("*")) + list(Path("logs").glob("*.log")):
        if path.is_file() and path.suffix != ".pt":
            z.write(path)
    z.write(DATA / "processed" / "nsclc_radiogenomics" / "build_log.csv", "build_log.csv")
print(archive, f"{archive.stat().st_size / 1e6:.1f} MB")
!du -sh runs/*/final_model.pt 2>/dev/null
!df -h ~ | tail -1

## Pasul următor (Etapa 3)

Registrarea PET pe CT, cuburile CT + PET și setul Lung-PET-CT-Dx — vezi PLAN.md, Etapa 3.
Datele brute NSCLC-Radiogenomics **rămân** pe server (sunt refolosite în etapele 3–5).